## Part 1



In [1]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
LLM = ChatGroq(model='openai/gpt-oss-20b')

In [3]:
result = LLM.invoke('Which LLM model are you?')

In [4]:
result.content

'I’m a large language model trained by OpenAI. I don’t have a single “model name” to give you, but I’m built on the latest generation of OpenAI’s GPT architecture.'

In [5]:
def groq_chat(prompt: str):
    res = LLM.invoke(prompt)
    return res
#Testing-
temp1 = groq_chat('Who is Elon Musk? in 60 words')
temp2 = groq_chat('Recipy of Bihari Chicken')
temp3 = groq_chat('What do you like the most?')

In [6]:
print(temp1,'\n',temp2,'\n',temp3)

content='Elon Musk is a South African‑born entrepreneur, inventor, and CEO of companies like Tesla, SpaceX, Neuralink, and X Holdings. He co‑founded PayPal and has pioneered electric vehicles, reusable rockets, and high‑speed transport. Known for ambitious vision, he aims to colonize Mars, build a hyperloop, and advance AI. His ventures shape technology, energy, and space industries worldwide for future generations today.' additional_kwargs={'reasoning_content': 'We need to answer: Who is Elon Musk? in 60 words. So we need to produce exactly 60 words. Let\'s count. We\'ll produce a concise description. 60 words. Let\'s write a paragraph about Elon Musk: entrepreneur, CEO of Tesla, SpaceX, Twitter, etc. Also mention his background. 60 words. Let\'s count manually.\n\nDraft: "Elon Musk is a South African‑born entrepreneur, inventor, and CEO of companies like Tesla, SpaceX, Neuralink, and X Holdings. He co‑founded PayPal and has pioneered electric vehicles, reusable rockets, and high‑spee

## Part 2 Grok + RAG


In [7]:
from langchain_core.output_parsers import StrOutputParser,PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import TextLoader
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter


docs = TextLoader('./obama.md',encoding='utf-8').load()
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=20)
chunks = splitter.split_documents(docs)
embedding = OllamaEmbeddings(model='nomic-embed-text')
vector_store = FAISS.from_documents(chunks,embedding)
retrieval = vector_store.as_retriever(search_kwargs={'k':3})

prompt = PromptTemplate(template='Give me very short detail from this context {query} only, do not use any other context',
                        input_variables=['query'])
LLM = ChatGroq(model='openai/gpt-oss-20b')
parser = StrOutputParser()

#LCEL Pipeline
chain = retrieval | prompt | LLM | parser

result = chain.invoke('Terrorism')
print(result)

C:\Users\shrey\AppData\Local\Temp\ipykernel_42876\4157422052.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Obama’s presidency was defined by the Affordable Care Act debate, wars in Iraq and Afghanistan, the 2011 bin Laden raid, a 2012 re‑election, the 2015 Paris climate and Iran nuclear agreements, the Supreme Court’s same‑sex marriage ruling, and persistent controversies over immigration, drone strikes, surveillance, and racial justice.


In [8]:
from pydantic import BaseModel,Field
from typing import Optional,Literal
class Structure(BaseModel):
    SystemInstruction: str = Field(description='Contains System Instruction Details Only in Short')
    Retrieved_context: str = Field(description='Contains Retrieved Context Only')
    User_question : str = Field(description='Contains User Questions Details Only')
parser_py = PydanticOutputParser(pydantic_object=Structure)
prompt_1 = PromptTemplate(template='Convert This Context {context} into this format {format}',
                          input_variables=['context'],
                          partial_variables={'format':parser_py.get_format_instructions()})
chain =  retrieval | prompt_1 | LLM | parser
result = chain.invoke('Who is Obama wife?')
print(result)

```json
{
  "SystemInstruction": "Process user query based on retrieved context.",
  "Retrieved_context": "His presidency revealed both the power and the limitations of political leadership. A president can influence national priorities, appoint officials, sign laws, and represent the country internationally. But lasting social change also depends on courts, legislatures, institutions, communities, and ordinary citizens.\n\n## Chapter 9: Beyond the Presidency\n\nOn January 20, 2017, Obama left office after completing two terms. Donald Trump succeeded him as president.\n\nThe transfer of power marked the end of an extraordinary chapter in American political history. Obama was no longer responsible for directing the federal government, but his influence continued through public discussion, political organizing, writing, and charitable work.\n\nAlongside his wife, Michelle Obama, he helped establish the Obama Foundation, which supports leadership development and civic engagement initiativ

Task 10: Observations & Insights Write short answers: 
1. Why Groq is suitable for real-time apps?
It uses LPU (Language Processing Unit) which gives response in very low time, It has low latency
2. Groq vs OpenAl latency comparison (conceptual)?
Groq result are much faster but based on model and hardware uses it may varies.
3. Benefits of API-first GenAI architecture?
It gives the AI logic from the user interface, making GenAI applications more flexible, reusable, scalable and easier to maintain.
